# Web Scraping with Python: Extracting Structured Data from Web Pages

## Project Overview

This notebook documents my practical introduction to **web scraping with Python**.

The work began with understanding how webpages are structured using **HTML (HyperText Markup Language)** and progressed to using Python to retrieve webpage content, parse HTML, locate specific elements, extract text, clean the extracted content, and convert a webpage table into a structured Pandas DataFrame.

The exercises covered three stages:

1. A guided practice page from **Scrape This Site**
2. A table-extraction exercise using **Worldometer**
3. A more complete scraping workflow using **Wikipedia**, where tabular data was extracted and exported to CSV.

## Learning Objectives

- Understand the role of HTML in webpage structure.
- Use `requests` to retrieve webpage content.
- Use `BeautifulSoup` to parse HTML.
- Locate elements with `find()` and `find_all()`.
- Extract text from HTML elements.
- Clean extracted text with `.strip()`.
- Identify and extract HTML tables.
- Convert scraped rows into a Pandas DataFrame.
- Export the resulting structured data to CSV.


## Tools & Libraries

- **Python**
- **Requests** — used to send HTTP requests and retrieve webpage content.
- **BeautifulSoup** — used to parse and navigate HTML.
- **Pandas** — used to structure the extracted table data into a DataFrame.
- **Jupyter Notebook** — used as the development environment.


## 1. Introduction to Web Scraping

Web scraping involves programmatically retrieving information from webpages and extracting specific content from the page.

Before extracting data, it is useful to understand the underlying HTML structure of a webpage, including tags, classes, tables, and other elements.


In [ ]:
from bs4 import BeautifulSoup
import requests


## 2. Guided Practice with Scrape This Site

The first exercise used the practice page on Scrape This Site.

The workflow was:

**URL → HTTP request → HTML response → BeautifulSoup parser → element selection → text extraction → text cleaning**


In [ ]:
url = "https://www.scrapethissite.com/pages/forms/"

page = requests.get(url)
Soup = BeautifulSoup(page.text, "html")


### Inspecting the HTML Structure

I used `find()` and `find_all()` to explore elements within the parsed HTML document.

This helped me understand how specific webpage elements can be located before extracting their content.


In [ ]:
Soup.find()


In [ ]:
Soup.find("div")


In [ ]:
Soup.find_all("div")


In [ ]:
print(Soup.prettify())


### Locating a Specific Element

I then targeted a paragraph element with the class `lead`.


In [ ]:
Soup.find("p", class_="lead")


In [ ]:
Soup.find_all("div", class_="col_md_12")


### Extracting and Cleaning Text

After locating the required element, I extracted only its text and then used `.strip()` to remove unnecessary whitespace.


In [ ]:
Soup.find("p", class_="lead").text


In [ ]:
Soup.find("p", class_="lead").text.strip()


## 3. Extracting Tables from Web Pages

The next exercises focused on extracting structured information from webpages containing HTML tables.

This introduced an important data-acquisition workflow: identifying the correct table, locating table headers, extracting table rows and cells, cleaning the values, and converting the result into a structured dataset.


### Worldometer: Largest Cities in the World

The Worldometer page was used to practice locating an HTML table and extracting its column headings.


In [ ]:
url = "https://www.worldometers.info/population/largest-cities-in-the-world/"

page = requests.get(url)
Soup = BeautifulSoup(page.text, "html")


In [ ]:
Soup.find("table")


In [ ]:
Soup.find_all("table")


In [ ]:
table = Soup.find("table")


### Extracting Table Headers

The table headers were located with the `<th>` elements. I first extracted the text and then cleaned the headings with `.strip()`.


In [ ]:
world_titles = Soup.find_all("th")
world_titles


In [ ]:
world_table_titles = [title.text for title in world_titles]
world_table_titles


In [ ]:
world_table_titles = [title.text.strip() for title in world_titles]
print(world_table_titles)


## 4. Full Table Extraction: U.S. Companies by Revenue

The most complete exercise used Wikipedia's **List of largest companies in the United States by revenue**.

The workflow expanded from simply locating HTML elements to building a structured dataset from the table.


In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_largest_companies_in_the_United_States_by_revenue"

headers = {
    "User-Agent": "Mozilla/5.0"
}

page = requests.get(url, headers=headers)
Soup = BeautifulSoup(page.text, "html.parser")


### Locating the Target Table

I inspected the available tables and selected the relevant table from the page.


In [ ]:
table = Soup.find("table")


In [ ]:
Soup.find_all("table")


In [ ]:
Soup.find_all("table")[0]


In [ ]:
Soup.find("table", class_="wikitable sortable")


### Extracting Column Names

The table headers were extracted from the `<th>` elements and cleaned before being used as DataFrame column names.


In [ ]:
world_titles = Soup.find_all("th")

world_table_titles = [title.text.strip() for title in world_titles]
print(world_table_titles)


### Creating the DataFrame Structure

Pandas was then used to create an empty DataFrame using the extracted table headers.


In [ ]:
import pandas as pd

df = pd.DataFrame(columns=world_table_titles)
df


### Extracting Table Rows

Each table row was inspected using `<tr>`, while the individual data cells were located using `<td>`.

The extracted text was cleaned with `.strip()`.


In [ ]:
column_data = table.find_all("tr")

for row in column_data[1:]:
    row_data = row.find_all("td")
    individual_row_data = [data.text.strip() for data in row_data]
    print(individual_row_data)


### Handling Rows with Different Lengths

During the extraction process, rows were checked before being added to the DataFrame.

- Empty rows were skipped.
- Rows with fewer values than the DataFrame columns were padded with `None`.
- Rows with more values were trimmed to the expected number of columns.


In [ ]:
for row in column_data[1:]:
    row_data = row.find_all("td")
    individual_row_data = [data.text.strip() for data in row_data]

    if not individual_row_data:
        continue

    if len(individual_row_data) < len(df.columns):
        individual_row_data.extend(
            [None] * (len(df.columns) - len(individual_row_data))
        )
    elif len(individual_row_data) > len(df.columns):
        individual_row_data = individual_row_data[:len(df.columns)]

    length = len(df)
    df.loc[length] = individual_row_data


### Inspecting the Scraped Dataset


In [ ]:
df


### Exporting the Scraped Data

The completed DataFrame was exported as a CSV file so that the scraped information could be used as a standalone dataset.

For a portfolio repository, the output should be saved using a relative project path rather than a personal computer directory.


In [ ]:
df.to_csv("usa_largest_companies.csv", index=False)


## 5. End-to-End Workflow

The web-scraping process covered in this notebook can be summarized as:

**1. Identify the webpage**  
↓  
**2. Send an HTTP request with `requests`**  
↓  
**3. Receive the HTML response**  
↓  
**4. Parse the HTML with BeautifulSoup**  
↓  
**5. Locate relevant tags, classes, and tables**  
↓  
**6. Extract text and table cells**  
↓  
**7. Clean extracted text with `.strip()`**  
↓  
**8. Structure the extracted records with Pandas**  
↓  
**9. Export the dataset to CSV**


## Key Learning Outcomes

This exercise strengthened my understanding of:

- HTML structure and webpage elements.
- HTTP requests in Python.
- HTML parsing with BeautifulSoup.
- Element selection using `find()` and `find_all()`.
- Text extraction and basic cleaning.
- HTML table extraction.
- Building structured datasets from scraped rows.
- Using Pandas to organize scraped data.
- Exporting data to CSV.

## Conclusion

This project marked my transition from Python fundamentals into practical **data acquisition**.

Instead of working only with pre-existing datasets, I learned how information can be retrieved from webpages, extracted from HTML, cleaned, structured, and saved for further analysis.

The next step in this workflow would be to take a scraped dataset and apply a complete data-analysis process, including further cleaning, exploratory analysis, visualization, and insight generation.
